In [55]:
import os
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

model = ChatOpenAI(
    model=os.getenv("LLM_MODEL_ID"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL"),
    temperature=0,
)

In [45]:
import os
from typing import Literal
from tavily import TavilyClient
from deepagents import create_deep_agent

tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))


def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Run a web search"""
    return tavily_client.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )


agent = create_deep_agent(model=model, tools=[internet_search])

agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is langgraph?",
            }
        ]
    }
)

{'messages': [HumanMessage(content='What is langgraph?', additional_kwargs={}, response_metadata={}, id='391df180-5b01-489b-b208-5bf4fd687c95'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 118, 'prompt_tokens': 5290, 'total_tokens': 5408, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 70, 'rejected_prediction_tokens': None, 'text_tokens': 118}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 5290}}, 'model_provider': 'openai', 'model_name': 'qwen3.5-plus-2026-02-15', 'system_fingerprint': None, 'id': 'chatcmpl-31c63327-0b34-965b-ad3f-4efc887216d4', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cf789-e105-7242-9d4a-c2a18c6c5d34-0', tool_calls=[{'name': 'internet_search', 'args': {'query': 'LangGraph what is it framework library', 'max_results': 5}, 'id': 'call_51d62314f81745a0bd12c2f5', 'type': 'too

In [ ]:
from langchain.tools import tool
from langchain.agents.middleware import wrap_tool_call
from deepagents import create_deep_agent


@tool
def get_weather(city: str) -> str:
    """Get the weather in a city."""
    return f"The weather in {city} is sunny."


call_count = [0]  # Use list to allow modification in nested function


@wrap_tool_call
def log_tool_calls(request, handler):
    """Intercept and log every tool call - demonstrates cross-cutting concern."""
    call_count[0] += 1
    tool_name = request.name if hasattr(request, "name") else str(request)

    print(f"[Middleware] Tool call #{call_count[0]}: {tool_name}")
    print(
        f"[Middleware] Arguments: {request.args if hasattr(request, 'args') else 'N/A'}"
    )

    # Execute the tool call
    result = handler(request)

    # Log the result
    print(f"[Middleware] Tool call #{call_count[0]} completed")

    return result


agent = create_deep_agent(
    model=model,
    tools=[get_weather],
    middleware=[log_tool_calls],
)

agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What's the weather in HangZhou?",
            }
        ]
    }
)

[Middleware] Tool call #1: ToolCallRequest(tool_call={'name': 'get_weather', 'args': {'city': 'HangZhou'}, 'id': 'call_4709fbafb0064d94a6279732', 'type': 'tool_call'}, tool=StructuredTool(name='get_weather', description='Get the weather in a city.', args_schema=<class 'langchain_core.utils.pydantic.get_weather'>, func=<function get_weather at 0x000001854643FBA0>), state={'messages': [HumanMessage(content="What's the weather in HangZhou?", additional_kwargs={}, response_metadata={}, id='330ff915-88ea-4bb7-a29c-a3c23a5e40aa'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 5237, 'total_tokens': 5313, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 44, 'rejected_prediction_tokens': None, 'text_tokens': 76}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 5237}}, 'model_provider': 'openai', 'model_name': '

{'messages': [HumanMessage(content="What's the weather in HangZhou?", additional_kwargs={}, response_metadata={}, id='330ff915-88ea-4bb7-a29c-a3c23a5e40aa'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 5237, 'total_tokens': 5313, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 44, 'rejected_prediction_tokens': None, 'text_tokens': 76}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 5237}}, 'model_provider': 'openai', 'model_name': 'qwen3.5-plus-2026-02-15', 'system_fingerprint': None, 'id': 'chatcmpl-3e5591d4-a135-9ddd-bdf7-dd1c7f698540', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cf739-36c3-7941-a90b-b067bf69be78-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'HangZhou'}, 'id': 'call_4709fbafb0064d94a6279732', 'type': 'tool_call'}], invalid_tool_calls=[], usage_me

In [18]:
import os
from typing import Literal
from tavily import TavilyClient
from deepagents import create_deep_agent

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Run a web search"""
    return tavily_client.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )


research_subagent = {
    "name": "research-agent",
    "description": "Used to research more in depth questions",
    "system_prompt": "You are a great researcher",
    "tools": [internet_search],
}
subagents = [research_subagent]

agent = create_deep_agent(model=model, subagents=subagents)

agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is langgraph?",
            }
        ]
    }
)

{'messages': [HumanMessage(content='What is langgraph?', additional_kwargs={}, response_metadata={}, id='1c4d1bf0-d2be-4e2e-8fa2-a99c159abb52'),
  AIMessage(content='**LangGraph** is a library for building **stateful, multi-actor applications with Large Language Models (LLMs)**. It\'s part of the LangChain ecosystem and is designed to help developers create complex, agentic workflows.\n\n## Key Features:\n\n1. **Graph-Based Architecture**: LangGraph lets you define workflows as graphs, where:\n   - **Nodes** represent actions or computations (like calling an LLM, running a tool, or processing data)\n   - **Edges** define the flow of control between nodes\n\n2. **State Management**: It maintains state across multiple steps of execution, which is essential for:\n   - Multi-turn conversations\n   - Long-running workflows\n   - Applications that need to remember context\n\n3. **Cycles and Loops**: Unlike simple linear chains, LangGraph supports cyclic graphs, enabling:\n   - Iterative refi

In [19]:
# By default we provide a StateBackend
agent = create_deep_agent()

# Under the hood, it looks like
from deepagents.backends import StateBackend

agent = create_deep_agent(
    backend=(lambda rt: StateBackend(rt))   # Note that the tools access State through the runtime.state
)

In [ ]:
from langchain.tools import tool
from deepagents import create_deep_agent
from langgraph.checkpoint.memory import MemorySaver


@tool
def delete_file(path: str) -> str:
    """Delete a file from the filesystem."""
    return f"Deleted {path}"


@tool
def read_file(path: str) -> str:
    """Read a file from the filesystem."""
    return f"Contents of {path}"


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email."""
    return f"Sent email to {to}"


# Checkpointer is REQUIRED for human-in-the-loop
checkpointer = MemorySaver()

agent = create_deep_agent(
    model=model,
    tools=[delete_file, read_file, send_email],
    interrupt_on={
        "delete_file": True,  # Default: approve, edit, reject
        "read_file": False,  # No interrupts needed
        "send_email": {"allowed_decisions": ["approve", "reject"]},  # No editing
    },
    checkpointer=checkpointer,  # Required!
)

In [25]:
from urllib.request import urlopen
from deepagents import create_deep_agent
from deepagents.backends.utils import create_file_data
from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()

skill_url = "https://raw.githubusercontent.com/langchain-ai/deepagents/refs/heads/main/libs/cli/examples/skills/langgraph-docs/SKILL.md"
with urlopen(skill_url) as response:
    skill_content = response.read().decode("utf-8")

skills_files = {"/skills/langgraph-docs/SKILL.md": create_file_data(skill_content)}

agent = create_deep_agent(
    model=model,
    skills=["/skills/"],
    checkpointer=checkpointer,
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is langgraph?",
            }
        ],
        # Seed the default StateBackend's in-state filesystem (virtual paths must start with "/").
        "files": skills_files,
    },
    config={"configurable": {"thread_id": "12345"}},
)
print(result["messages"][-1].content)

Based on my knowledge, **LangGraph** is a powerful framework for building complex, stateful AI applications and multi-agent systems. Here's what it is:

## What is LangGraph?

LangGraph is a library built on top of LangChain that enables you to create **stateful, multi-actor applications** using a graph-based approach. Instead of simple linear chains, it allows you to build workflows with cycles, branches, and persistent memory.

## Key Features:

1. **Graph-Based Architecture**: Model your AI workflows as nodes (actions/states) and edges (transitions), making complex flows visual and manageable.

2. **State Management**: Maintains state across multiple steps, allowing agents to remember context and build upon previous interactions.

3. **Cycles & Loops**: Unlike linear chains, LangGraph supports cyclic graphs, enabling iterative processes like refinement loops, multi-turn conversations, and retry mechanisms.

4. **Multi-Agent Systems**: Coordinate multiple specialized agents that can 

In [26]:
from urllib.request import urlopen

from deepagents import create_deep_agent
from deepagents.backends.utils import create_file_data
from langgraph.checkpoint.memory import MemorySaver

with urlopen(
    "https://raw.githubusercontent.com/langchain-ai/deepagents/refs/heads/main/examples/text-to-sql-agent/AGENTS.md"
) as response:
    agents_md = response.read().decode("utf-8")
checkpointer = MemorySaver()

agent = create_deep_agent(
    model=model,
    memory=["/AGENTS.md"],
    checkpointer=checkpointer,
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Please tell me what's in your memory files.",
            }
        ],
        # Seed the default StateBackend's in-state filesystem (virtual paths must start with "/").
        "files": {"/AGENTS.md": create_file_data(agents_md)},
    },
    config={"configurable": {"thread_id": "123456"}},
)
print(result["messages"][-1].content)

Based on my memory files, here's what I have stored:

**AGENTS.md** - This contains my core instructions as a Text-to-SQL Agent:

**My Role:**
- I'm designed to interact with a SQL database (specifically a SQLite Chinook database)
- I explore tables, examine schemas, generate SQL queries, execute them, and format results clearly

**Database:**
- SQLite Chinook database containing data about a digital media store
- Includes tables for: artists, albums, tracks, customers, invoices, employees

**Query Guidelines:**
- Limit results to 5 rows by default
- Order results by relevant columns
- Only query relevant columns (not SELECT *)
- Double-check SQL syntax
- Rewrite queries if they fail

**Safety Rules:**
- READ-ONLY access - only SELECT queries allowed
- NEVER execute: INSERT, UPDATE, DELETE, DROP, ALTER, TRUNCATE, CREATE

**Planning Approach:**
- For complex questions, I should use the `write_todos` tool to break down tasks
- Plan which tables to examine and query structure
- Execute an